# Component protocols

A reusable component can supply a decomposition, resource declarations, numerical tensors, classical actions and presentation metadata. FlagQuantum distributes these responsibilities among its existing modules. Core records hold signatures and owned values. Compiler transforms the stored Program. Simulation evaluates numerical data. Drawer and Ecosystem provide presentation and external translation.

## Components of a protocol

The caller selects a domain-owned entry function. A component author supplies an immutable body, a declaration or an explicit callback. Domain helpers connect that implementation to its consumer. A fallback uses already-declared component structure when that domain supports it. Missing implementations retain the owning domain's diagnostic.

A signature declares an interface. It does not supply a decomposition, numerical evaluator or hardware implementation. Supplying a custom tensor or a resource estimate independently does not imply that the other protocols are implemented.

This notebook maps both Markdown cells of the complete locked protocol overview. Its source revision is `bcb17bb25360ed68ce3bc084645db3e929799389`, Git blob `8fe215cc166ccc223aeeb4edfa0dccfeec37ffb2`. Copyright 2023 Google LLC, Apache-2.0. Documentation is adapted to FlagQuantum's domain boundaries.


## Protocol entry points

### Decomposition

Construct a typed Program with `ComponentProgramBuilder`. Calls refer to existing FunctionRecord definitions and return fresh owned values. `expand_component_calls` handles stored decompositions. `bind_register_program` assigns physical wires and workspace. Ecosystem's `import_cirq_register_gate` and `import_cirq_register_optree` translate admitted external decompositions into the same records. An unavailable body retains the compiler's explicit refusal.

The existing register example supplies a complete body. Its direct calls, physical stream and ordinary append entry can be inspected together.

```python
from collections import Counter

import flagquantum as fq
from examples.algorithms.register_gates import RegisterControlledTargets
from flagquantum.compiler._component_trace.program_call_graph import (
    program_component_call_graph,
)
from flagquantum.drawer.component_score import component_register_score
from flagquantum.simulation.unitary import get_unitary

operation = RegisterControlledTargets((0,), (1, 2))
program = operation.program()
bound = operation.bind()
call_graph = program_component_call_graph(program)
score = component_register_score(program)
primitive_counts = dict(Counter(gate.name for gate in bound.circuit.instructions))
assert primitive_counts == {"cx": 2}
assert call_graph.leaf_counts["cx"] == 2
circuit = fq.Circuit(3)
circuit.append(operation)
operator = get_unitary(bound.circuit)
```

### Call Graph

`program_component_call_graph` counts actual stored call occurrences. Its fallback derives direct callees from a stored decomposition and delegates weighted traversal to `symbolic_call_graph`. Explicit declarations can independently provide symbolic resource models. Rewrites, stopping predicates and depth cutoffs belong to the graph query. Physical primitive counts and analytical estimates are recorded separately. The example's two emitted CX instructions agree with its two CX call leaves. Split/join bookkeeping remains visible in the stored graph.

See [the complete call-graph tutorial](call_graph_protocol.ipynb) for explicit symbolic declarations and resource-model fallbacks.

### Tensor

Simulation's `ComponentTensorProviders` associates declared function names with numerical suppliers. Compiler's `expand_components_for_tensor` or `expand_component_root_for_tensor` retains those leaves and expands available bodies. `contract_component_tensors` contracts the resulting nodes. `get_unitary` in the example above evaluates the bound circuit's complete operator for a small diagnostic. Tensor production and physical decomposition remain separate capabilities.

The source overview names an `add_my_tensors` author method. The locked implementation exposes `my_tensors` instead. The native equivalent uses explicit suppliers and the existing tensor entry functions. This documentation preserves that source discrepancy in its provenance rather than introducing an extra author method.

### Classical

`ComponentClassicalState` evaluates selected typed calls with explicit value and phase callbacks. The existing evaluator can recurse through declared components. A non-classical operation must supply a valid numerical representation for the requested mode or produce the owning domain's failure. A truth table alone does not determine coherent phases.

The existing classical/tensor example composes both representations through the same native component and independent codec owners.

```python
import torch

from examples.algorithms.classical_tensor_crosscheck import (
    contract_classical_crosscheck,
    crosscheck_values,
)

values = {"ctrl": 1, "target": 0}
assert crosscheck_values("cnot", values) == {"ctrl": 1, "target": 1}
overlap = contract_classical_crosscheck("cnot", values)
torch.testing.assert_close(overlap, torch.ones_like(overlap), rtol=0, atol=2e-12)
```

### Drawing and interoperability

`component_register_score` projects the typed call graph for Drawer without changing execution. Ecosystem's Cirq register exporters translate admitted bound operations, including the existing ownership and feedback contracts. These adapters retain their supported-domain checks. The external SDK object stays at the adapter boundary.

The graph and tensor examples above reuse existing production implementations. Wider classical-word Program admission remains pending Proposal 108. The original overview's universal Bloq method surface becomes separate domain entry functions and explicit callbacks. Existing complete interface, call-graph and tensor audits retain detailed source failures and unsupported boundaries.
